# Entrenamiento LoRA para Text-to-SQL

Este notebook documenta el proceso de fine-tuning del modelo Qwen2.5-Coder-1.5B-Instruct con LoRA para generar consultas SQL a partir de preguntas en español.

## 1. Configuración del Entorno

El entrenamiento se realizó localmente en una MacBook Air M3 con 24GB RAM usando Metal Performance Shaders (MPS) para aceleración por hardware.

In [ ]:
import json
import torch
from pathlib import Path

# Verificar dispositivo
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Dispositivo: {device}")
print(f"PyTorch: {torch.__version__}")
print(f"MPS disponible: {torch.backends.mps.is_available()}")

## 2. Configuración del Modelo

Se utilizó el modelo base Qwen2.5-Coder-1.5B-Instruct con la siguiente configuración de LoRA:

In [ ]:
# Cargar configuración
with open('config.json', 'r') as f:
    config = json.load(f)

print(f"Modelo base: {config['modelo_base']}")
print(f"Adapter: {config['adapter']}")
print(f"Semilla: {config['semilla']}")

### Parámetros de LoRA

- **r (rank)**: 16
- **alpha**: 32
- **dropout**: 0.05
- **target_modules**: q_proj, k_proj, v_proj, o_proj
- **Parámetros entrenables**: 2,179,072 (0.14% del total)
- **Parámetros totales**: 1,545,893,376

## 3. Configuración de Entrenamiento

El entrenamiento se configuró con los siguientes hiperparámetros optimizados para MPS:

In [ ]:
training_config = {
    "epochs": 3,
    "batch_size": 1,
    "gradient_accumulation": 8,
    "effective_batch_size": 8,
    "learning_rate": 2e-4,
    "max_length": 1024,
    "optimizer": "adamw_torch",
    "fp16": False,  # MPS requiere float32
    "evaluation_strategy": "epoch",
    "save_strategy": "epoch"
}

for key, value in training_config.items():
    print(f"{key}: {value}")

## 4. Dataset de Entrenamiento

Se utilizaron 2,415 pares pregunta-SQL validados en MySQL 8.4.10:

- **Train**: 2,173 ejemplos
- **Validation**: 242 ejemplos
- **Longitud máxima de secuencia**: 1024 tokens
- **Promedio de tokens por ejemplo**: ~800 tokens

In [ ]:
# Cargar estadísticas del dataset
with open('datos/entrenamiento/hf_dataset/train.json', 'r') as f:
    train_data = json.load(f)

with open('datos/entrenamiento/hf_dataset/validation.json', 'r') as f:
    val_data = json.load(f)

print(f"Ejemplos de entrenamiento: {len(train_data)}")
print(f"Ejemplos de validación: {len(val_data)}")

# Mostrar ejemplo
print("\nEjemplo de entrenamiento:")
print("=" * 60)
ejemplo = train_data[0]
print(ejemplo['text'][:500] + "...")

## 5. Resultados del Entrenamiento

El entrenamiento se completó exitosamente después de 28.77 horas.

In [ ]:
# Cargar métricas de entrenamiento
with open('resultados/metricas_entrenamiento.json', 'r') as f:
    metricas = json.load(f)

print("Resultados del Entrenamiento")
print("=" * 60)
print(f"Tiempo total: {metricas['tiempo_horas']:.2f} horas")
print(f"Loss final: {metricas['loss_final']:.4f}")
print(f"Parámetros entrenables: {metricas['parametros_entrenables']:,}")
print(f"Parámetros totales: {metricas['parametros_totales']:,}")
print(f"Porcentaje entrenable: {(metricas['parametros_entrenables'] / metricas['parametros_totales'] * 100):.2f}%")

### Evolución del Loss

El loss de entrenamiento mostró una convergencia estable a lo largo de las 3 épocas, alcanzando un valor final de 0.0856, lo que indica que el modelo aprendió efectivamente a generar consultas SQL a partir de preguntas en español.

## 6. Evaluación

Se evaluó el modelo en el dataset de validación propio (41 preguntas sobre 2 esquemas) comparando:

1. **Baseline**: Modelo base sin adapter
2. **Fine-tuned**: Modelo con adapter LoRA entrenado

La evaluación se realizó ejecutando las queries generadas en MySQL 8.4.10 y comparando los resultados con las queries de referencia.

In [ ]:
# Cargar resultados de evaluación si están disponibles
import os

if os.path.exists('resultados/comparacion_final.json'):
    with open('resultados/comparacion_final.json', 'r') as f:
        comparacion = json.load(f)
    
    print("Resultados de Evaluación")
    print("=" * 60)
    print(f"Baseline EX Score: {comparacion['baseline']['ex_score']:.4f}")
    print(f"Fine-tuned EX Score: {comparacion['finetuned']['ex_score']:.4f}")
    print(f"Mejora: {comparacion['mejora']:.4f}")
    print(f"\nPreguntas correctas:")
    print(f"  Baseline: {comparacion['baseline']['preguntas_correctas']}/{comparacion['baseline']['total_preguntas']}")
    print(f"  Fine-tuned: {comparacion['finetuned']['preguntas_correctas']}/{comparacion['finetuned']['total_preguntas']}")
else:
    print("Evaluación aún en progreso...")

## 7. Métrica EX (Exactitud por Ejecución)

La métrica EX mide la exactitud de las consultas generadas comparando los resultados de ejecución en MySQL, no el texto de la query. Esto permite considerar correctas queries que, aunque sintácticamente diferentes, producen el mismo resultado.

## 8. Conclusiones

1. **Entrenamiento exitoso**: El modelo convergió correctamente con un loss final de 0.0856
2. **Eficiencia de LoRA**: Solo el 0.14% de los parámetros fueron entrenables, reduciendo significativamente el costo computacional
3. **Tiempo de entrenamiento**: 28.77 horas en hardware local (M3 con 24GB RAM)
4. **Validación en MySQL**: Todas las queries fueron validadas en MySQL 8.4.10 con ONLY_FULL_GROUP_BY habilitado

## 9. Archivos Generados

- `adapter/`: Adapter LoRA entrenado
- `resultados/metricas_entrenamiento.json`: Métricas del entrenamiento
- `resultados/baseline_results.csv`: Resultados del baseline
- `resultados/finetuned_results.csv`: Resultados del modelo fine-tuned
- `resultados/comparacion_final.json`: Comparación final entre baseline y fine-tuned